# ScreamingFace · create a custom benchmark

Turn a small set of researcher-owned examples into a benchmark that any Fusion can evaluate. The
boundary is intentionally compact:

1. prepare ordinary `sf.Case` values;
2. choose one grader and one aggregator;
3. construct an immutable `sf.Benchmark`.

The default notebook is entirely local. It needs no Docker, provider credentials, Hugging Face
access, or network connection.

## 1 · Define the cases

In [ ]:
import screamingface as sf

cases = [
    sf.Case(
        "astronomy-1",
        (
            "Which planet is closest to the Sun?\n\n"
            "A. Venus\nB. Mercury\nC. Mars\nD. Earth\n\n"
            "Reply with only A, B, C, or D."
        ),
        reference="B",
        metadata={"topic": "astronomy"},
    ),
    sf.Case(
        "biology-1",
        (
            "Which organelle produces most cellular ATP?\n\n"
            "A. Nucleus\nB. Ribosome\nC. Mitochondrion\nD. Lysosome\n\n"
            "Reply with only A, B, C, or D."
        ),
        reference="C",
        metadata={"topic": "biology"},
    ),
    sf.Case(
        "physics-1",
        (
            "What is the SI unit of force?\n\n"
            "A. Newton\nB. Joule\nC. Watt\nD. Pascal\n\n"
            "Reply with only A, B, C, or D."
        ),
        reference="A",
        metadata={"topic": "physics"},
    ),
]

len(cases)

Each case has four deliberately small fields:

- `id` is a stable unique identity used to pair results;
- `input` is exactly what every Fusion member receives;
- `reference` is the local grading target; and
- `metadata` is optional researcher-owned annotation.

The reference is sealed from execution: the researcher and local grader can read it, but the
reference never enters a model request.

## 2 · Inspect your own case values

In [ ]:
{
    "id": cases[0].id,
    "input": cases[0].input,
    "reference": cases[0].reference,
    "metadata": cases[0].metadata,
}

These are your ordinary Python values, so inspect the `cases` list you created.
ScreamingFace does not add a second case browser or iteration DSL.

## 3 · Assemble the benchmark

In [ ]:
benchmark = sf.Benchmark(
    "tiny-science@1",
    title="Tiny Science",
    cases=cases,
    grader=sf.graders.ExactChoice(),
    aggregator=sf.aggregators.Mean(),
)

benchmark

The version is part of the opaque benchmark ID, so changing the cases or scoring
contract can produce a new identity such as `tiny-science@2`.

`ExactChoice()` selects the deterministic grader contract. `Mean()` selects paired Recipe/member
accuracy, best-member baseline, and gain. When this benchmark is registered on an engine, those
strategies become grader and aggregator routes inside the complete URL4 run.

## 4 · Inspect the public definition

In [ ]:
{
    "id": benchmark.id,
    "title": benchmark.title,
    "grader": benchmark.grader,
    "aggregator": benchmark.aggregator,
    "tools": benchmark.tools,
    "case_count": len(cases),
}

## 5 · Keep source loading and cleaning outside ScreamingFace

For a larger source, pass a zero-argument loader instead of an in-memory sequence:

```python
def load_cases():
    rows = read_my_source()  # Researcher-owned loading and cleaning
    return [
        sf.Case(
            row["id"],
            row["rendered_input"],
            reference=row["reference"],
            metadata=row.get("metadata"),
        )
        for row in rows
    ]

benchmark = sf.Benchmark(
    "my-benchmark@1",
    cases=load_cases,
    grader=sf.graders.ExactChoice(),
)
```

The loader, files, schemas, joins, and cleanup remain ordinary research code.
ScreamingFace starts at validated `sf.Case` values; it does not become an ETL framework.

## 6 · Declare required tools on the benchmark

If answering every case genuinely requires web research, declare that requirement once:

```python
research_benchmark = sf.Benchmark(
    "my-research-benchmark@1",
    cases=cases,
    grader=sf.graders.ExactChoice(),
    tools=(
        sf.tools.WebSearch(max_results=5),
        sf.tools.WebFetch(),
    ),
    max_tool_calls=8,
)
```

The engine then requires every answer-producing member to support the capability and adds it to
their model calls. The explicit tool-call budget prevents an unbounded agent loop. Tools
do not belong in individual model parameters, and credentials never belong in this definition.

Because this researcher-authored definition has no immutable engine policy route yet, its portable
policy is represented inline when an engine profile registers it. An official benchmark manifest
instead points to one versioned policy data route that every answer-producing member shares. Both
forms express the same provider-neutral behavior; neither selects Tavily or OpenRouter.

## 7 · Register it before execution

This typed definition is authoring input for an engine deployment; it is not an
upload API. To execute it as one reproducible URL4 today, add its case, grader, and aggregator
routes to a ScreamingFace engine profile. Once the engine advertises its manifest,
`sf.benchmarks.load("tiny-science@1")` returns the executable benchmark.

Keeping registration explicit prevents the client from silently falling back to local case loops
that cannot be represented by the shared run URL4.

## Recap

- prepare plain `sf.Case` values with stable IDs;
- keep each `input` exact and each `reference` sealed from model requests;
- choose a grader and aggregator explicitly;
- keep source loading and cleaning in researcher-owned code;
- put shared tool requirements on the benchmark; and
- register the definition on an engine before evaluating it with any Recipe.

This is the complete custom-benchmark boundary—small enough for three handwritten cases and
flexible enough for a loader that produces thousands.